# Notebook 13: Fine-tuning with Unsloth - SOLUTION

**This is the SOLUTION notebook with complete implementations.**

In [11]:
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
# Model configuration
model_name = "/data/Ministral-3-14B-Instruct-2512"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()


Model found locally.


NameError: name 'FastLanguageModel' is not defined

In [ ]:
# Load and format dataset
local_model_path = Path("/data/alpaca-cleaned")
if local_model_path.is_dir():
    print("Data found locally.")
    dataset = load_dataset("/data/alpaca-cleaned", split="train")
else:
    from dotenv import load_dotenv
    print("Data not found locally. Downloading from Hugging Face...")
    
    load_dotenv(Path(".env"))
    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN not set. Add it to your .env file: HF_TOKEN=hf_your_token_here")
    
    dataset = load_dataset("/unsloth/alpaca-cleaned", split="train")
    print("saving dataset locally...")
    dataset.save_to_disk("/data/alpaca-cleaned")

def format_example(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    
    user_message = instruction
    if input_text and input_text.strip():
        user_message = f"{instruction}\n\n{input_text}"
    
    text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant.<|eot_id|>"
    text += f"<|start_header_id|>user<|end_header_id|>\n\n{user_message}<|eot_id|>"
    text += f"<|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
    
    return {"text": text}

dataset = dataset.map(format_example)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
val_dataset = split['test']

print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")

Training examples: 49,172
Validation examples: 2,588


## Exercise 13.1: Configure Training Arguments - SOLUTION

In [ ]:
####solution 13.1

# TODO: Configure training arguments
#
training_args = TrainingArguments(
    output_dir="./outputs",  # Where to save checkpoints
     
    # Batch size settings
    per_device_train_batch_size=4,  # Start with 4, adjust based on memory
    gradient_accumulation_steps=4,  # 4 gives effective batch size of 16
     
    # Learning rate settings
    learning_rate=2e-4,  # Try 2e-4 for LoRA
    lr_scheduler_type="cosine",  # "cosine" or "linear"
    warmup_ratio=0.1,  # 0.05-0.1
     
    # Training duration
    num_train_epochs=3,  # 1-3 epochs
    # OR use max_steps for fixed step training
    # max_steps=1000,
     
    # Logging and saving
    logging_steps=10,  # Log every N steps
    save_steps=100,  # Save checkpoint every N steps
    save_total_limit=3,  # Keep only N most recent checkpoints
     
    # Evaluation
    eval_strategy="steps",
    eval_steps=100,  # Evaluate every N steps
     
    # Performance settings
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",  # Memory-efficient optimizer
    weight_decay=0.01,
     
    # Misc
    seed=42,
    report_to="none",  # Or "wandb" if you want logging
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [5]:
####test cell
effective_batch_size = (
    training_args.per_device_train_batch_size * 
    training_args.gradient_accumulation_steps
)
total_steps = len(train_dataset) // effective_batch_size * training_args.num_train_epochs

print(f"Training Configuration:")
print(f"  Effective batch size: {effective_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Total epochs: {training_args.num_train_epochs}")
print(f"  Estimated total steps: {total_steps:,}")
print(f"  Warmup steps: {int(total_steps * training_args.warmup_ratio)}")

Training Configuration:
  Effective batch size: 16
  Learning rate: 0.0002
  Total epochs: 1
  Estimated total steps: 3,073
  Warmup steps: 153


## Exercise 13.2: Create Trainer - SOLUTION

In [ ]:
####solution 13.2
 # TODO: Create the SFTTrainer
#
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # Optional but recommended
    args=training_args,
     
    # Data configuration
    dataset_text_field="text",  # Column containing formatted text
    max_seq_length=max_seq_length,
    packing=False,  # Set True to pack multiple examples per sequence
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/49172 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/2588 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [7]:
####test cell
if torch.cuda.is_available():
    print(f"GPU memory before training:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

GPU memory before training:
  Allocated: 10.14 GB
  Reserved: 10.39 GB


## Exercise 13.3: Train the Model - SOLUTION

In [ ]:
####solution 13.3
# TODO: Start training
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,172 | Num Epochs = 1 | Total steps = 3,074
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 69,992,448 of 14,015,024,128 (0.50% trained)


Step,Training Loss,Validation Loss


In [ ]:
####plot cell
# Plot training metrics
import matplotlib.pyplot as plt

history = trainer.state.log_history

train_losses = [(h['step'], h['loss']) for h in history if 'loss' in h]
eval_losses = [(h['step'], h['eval_loss']) for h in history if 'eval_loss' in h]

plt.figure(figsize=(10, 4))

if train_losses:
    steps, losses = zip(*train_losses)
    plt.plot(steps, losses, label='Train Loss')

if eval_losses:
    steps, losses = zip(*eval_losses)
    plt.plot(steps, losses, label='Eval Loss', marker='o')

plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.show()

## Exercise 13.4: Save the Model - SOLUTION

In [ ]:
####solution 13.4
# TODO: Save the LoRA adapter weights
#
# Option 1: Save just the LoRA adapter (recommended)
model.save_pretrained("my_lora_adapter")
tokenizer.save_pretrained("my_lora_adapter")
#
# Option 2: Save merged model (LoRA merged into base weights)
# model.save_pretrained_merged("my_merged_model", tokenizer, save_method="merged_16bit")

In [ ]:
####test cell
import os

save_dir = "my_lora_adapter"
if os.path.exists(save_dir):
    files = os.listdir(save_dir)
    print(f"Saved files in {save_dir}:")
    for f in files:
        size = os.path.getsize(os.path.join(save_dir, f)) / 1e6
        print(f"  {f}: {size:.2f} MB")

## Exercise 13.5: Test the Fine-tuned Model - SOLUTION

In [ ]:
####test cell
FastLanguageModel.for_inference(model)

test_prompts = [
    "Explain quantum computing in simple terms.",
    "Write a short poem about coding.",
    "What are three tips for learning a new programming language?",
]

In [ ]:
####solution 13.5
 # TODO: Generate responses for test prompts

for prompt in test_prompts:
    # Format prompt using Llama 3 template
    formatted = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant.<|eot_id|>"
    formatted += f"<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>"
    formatted += f"<|start_header_id|>assistant<|end_header_id|>\n\n"
     
    # Tokenize
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    # Decode and print
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print("-" * 50)

In [ ]:
####test cell
# Final memory check
if torch.cuda.is_available():
    print(f"\nFinal GPU memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")